# nanochat-d20 RL Training Analysis

Compares three reward systems on GSM8K:
- **Baseline** (`rl-gsm8k-nanochat-d20`): exact-match only (1.0 if `#### <number>` matches, else 0)
- **Numeric Distance** (`…-numeric_distance`): exact-match + dense shaping via `exp(−|pred−ref|/(|ref|+1))`
- **Completion Brevity** (`…-completion_brevity`): exact-match + partial credit for parse + reasoning words + conciseness

Eval logs must be in `./eval_logs/<run_name>/eval_step_XXXXXX.json`.
Download with: `uv run modal run nanochat_modal.py::download_all_d20_eval_logs`

In [ ]:
import json, re, os, glob, warnings
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

EVAL_LOG_DIR = Path('eval_logs')

RUNS = {
    'Baseline':           'rl-gsm8k-nanochat-d20',
    'Numeric Distance':   'rl-gsm8k-nanochat-d20-numeric_distance',
    'Completion Brevity': 'rl-gsm8k-nanochat-d20-completion_brevity',
}

COLORS = {
    'Baseline':           '#2196F3',
    'Numeric Distance':   '#FF5722',
    'Completion Brevity': '#4CAF50',
    'Karpathy':           '#9C27B0',
}

In [ ]:
# ── Data Loading ──────────────────────────────────────────────────────────────

def load_run(run_dir: Path) -> list[dict]:
    """Load all eval_step_*.json files for one run, sorted by step."""
    files = sorted(run_dir.glob('eval_step_*.json'))
    steps = []
    for f in files:
        with open(f) as fh:
            data = json.load(fh)
        steps.append(data)
    return steps

runs: dict[str, list[dict]] = {}
for label, run_name in RUNS.items():
    run_dir = EVAL_LOG_DIR / run_name
    if run_dir.exists():
        runs[label] = load_run(run_dir)
        print(f'{label:25s}: {len(runs[label])} eval snapshots'
              f'  (steps: {[s["step"] for s in runs[label]]})')
    else:
        print(f'{label:25s}: *** not found at {run_dir} ***')

if not runs:
    raise RuntimeError('No eval logs found. Run download_all_d20_eval_logs first.')

In [ ]:
# ── Flatten records into DataFrame ────────────────────────────────────────────

rows = []
for label, snapshots in runs.items():
    for snap in snapshots:
        step = snap['step']
        for rec in snap.get('records', []):
            # pass@k: outcome[i]['is_correct'] for the i-th sample
            outcomes = rec.get('outcomes', [])
            for k_idx, outcome in enumerate(outcomes):
                rows.append({
                    'run':       label,
                    'step':      step,
                    'idx':       rec['idx'],
                    'question':  rec['question'],
                    'reference': rec['reference'],
                    'sample':    k_idx,
                    'is_correct': outcome['is_correct'],
                    'generated': outcome['generated_text'],
                })

df = pd.DataFrame(rows)
print(f'Total rows: {len(df):,}  |  runs: {df["run"].unique()}  |  steps: {sorted(df["step"].unique())}')
df.head(2)

## 1. Pass@k Curves

In [ ]:
# Karpathy digitised reference curves (pass@1 and pass@8 from the nanochat-d32 RL paper figure)
# Replace these with your actual digitised values if you have them.
KARPATHY_STEPS  = [0, 60, 120, 180, 240, 300, 360, 420, 480, 540, 600, 650]
KARPATHY_PASS1  = [0.04, 0.08, 0.12, 0.16, 0.19, 0.22, 0.24, 0.26, 0.27, 0.28, 0.29, 0.30]
KARPATHY_PASS8  = [0.12, 0.22, 0.30, 0.36, 0.41, 0.44, 0.47, 0.49, 0.51, 0.52, 0.53, 0.54]

def passk_curve(df_run: pd.DataFrame, k: int) -> tuple[list, list]:
    """Return (steps, pass@k) for a single run's DataFrame."""
    out_steps, out_vals = [], []
    for step, grp in df_run.groupby('step'):
        # pass@k: for each problem, did any of the first k samples succeed?
        per_problem = grp[grp['sample'] < k].groupby('idx')['is_correct'].any()
        out_steps.append(step)
        out_vals.append(per_problem.mean())
    return out_steps, out_vals

for k in [1, 8]:
    fig, ax = plt.subplots(figsize=(9, 4.5))
    for label, snapshots in runs.items():
        df_run = df[df['run'] == label]
        steps, vals = passk_curve(df_run, k)
        ax.plot(steps, vals, 'o-', color=COLORS[label], label=label, linewidth=2, markersize=6)
    # Karpathy reference
    ref_vals = KARPATHY_PASS1 if k == 1 else KARPATHY_PASS8
    ax.plot(KARPATHY_STEPS, ref_vals, '--', color=COLORS['Karpathy'],
            label='Karpathy d32 (reference)', linewidth=1.5, alpha=0.7)
    ax.set_xlabel('RL Training Step')
    ax.set_ylabel(f'Pass@{k}')
    ax.set_title(f'GSM8K Pass@{k} — nanochat-d20 RL (400 test problems)')
    ax.legend()
    ax.set_ylim(0, None)
    plt.tight_layout()
    plt.savefig(f'passk_{k}_curve.png', bbox_inches='tight')
    plt.show()

## 2. Problem-Type Classifier

In [ ]:
PROBLEM_TYPES = {
    'money/commerce':  r'\$|dollar|cent|price|cost|paid|spend|bought|profit|sold|discount|tax|wage|earn|revenue|store|shop',
    'rate/unit':       r'per\s+\w+|mph|km/h|miles?\s+per|speed|rate|km\b|meter|litr|gallon|pound\b|ounce|feet|inch|yard|hour|minute|second\s+per',
    'percentage':      r'percent|%|fraction of|ratio|out of \d+|proportion',
    'counting':        r'how many|number of|count|total (people|students|boys|girls|children|items|animals|books|cars|players)',
    'geometry':        r'area|perimeter|volume|radius|diameter|triangle|rectangle|circle|square|angle|length|width|height',
    'fractions':       r'half|quarter|third|fraction|numerator|denominator|\d+/\d+',
    'age/time':        r'\bage\b|years? old|born|birthday|hours?\s+(ago|later)|days?\s+(ago|later)|week|month|decade|century',
}

def classify_problem(question: str) -> str:
    q = question.lower()
    for ptype, pattern in PROBLEM_TYPES.items():
        if re.search(pattern, q):
            return ptype
    return 'other'

# Add to dataframe (classify once per unique question)
qmap = {q: classify_problem(q) for q in df['question'].unique()}
df['problem_type'] = df['question'].map(qmap)

# Distribution
ptype_counts = df[df['step'] == df['step'].max()].drop_duplicates('idx')['problem_type'].value_counts()
fig, ax = plt.subplots(figsize=(8, 3.5))
ptype_counts.plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')
ax.set_title('Problem-Type Distribution (400 GSM8K test problems)')
ax.set_ylabel('Count')
ax.set_xlabel('')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig('problem_type_dist.png', bbox_inches='tight')
plt.show()
print(ptype_counts.to_string())

## 3. Error-Type Classifier

In [ ]:
def extract_numeric(text: str) -> float | None:
    """Extract the numeric answer from a generated response."""
    # Prefer #### format
    m = re.search(r'####\s*([\-\d,\.]+)', text)
    if m:
        try: return float(m.group(1).replace(',', ''))
        except ValueError: pass
    # Fallback: last number in text
    nums = re.findall(r'[\-]?\d+(?:[,\.]\d+)*', text)
    if nums:
        try: return float(nums[-1].replace(',', ''))
        except ValueError: pass
    return None

def extract_ref_numeric(ref: str) -> float | None:
    m = re.search(r'####\s*([\-\d,\.]+)', ref)
    if m:
        try: return float(m.group(1).replace(',', ''))
        except ValueError: pass
    nums = re.findall(r'[\-]?\d+(?:[,\.]\d+)*', ref)
    if nums:
        try: return float(nums[-1].replace(',', ''))
        except ValueError: pass
    return None

REASONING_WORDS = re.compile(
    r'\b(first|then|therefore|so|total|finally|thus|hence|because|since|next|step|now)\b', re.I)

def classify_error(generated: str, reference: str, is_correct: bool) -> str:
    if is_correct:
        return 'correct'
    gen = generated.strip()
    # Truncated: no #### marker at all and text ends mid-sentence
    has_marker = bool(re.search(r'####', gen))
    if not has_marker and len(gen) < 60:
        return 'truncated'
    if not has_marker:
        return 'missing_marker'
    pred = extract_numeric(gen)
    ref  = extract_ref_numeric(reference)
    if pred is None:
        return 'missing_marker'
    if ref is None:
        return 'wrong_arithmetic'
    rel_err = abs(pred - ref) / (abs(ref) + 1e-6)
    if rel_err < 0.02:          # within 2% but not exact (float edge case)
        return 'off_by_one'
    if rel_err < 0.5:           # ballpark but off
        return 'wrong_arithmetic'
    # Large deviation — check for hallucinated numbers in chain of thought
    cot_nums = re.findall(r'\d+(?:[,\.]\d+)?', gen)
    if len(cot_nums) > 8:       # many spurious numbers → hallucinated calculation
        return 'hallucinated_calc'
    return 'wrong_setup'

df['error_type'] = df.apply(
    lambda r: classify_error(r['generated'], r['reference'], r['is_correct']), axis=1)

# Summary on last eval step across all runs
last_step_df = df[df['step'] == df.groupby('run')['step'].transform('max')]
print(last_step_df.groupby(['run','error_type']).size().unstack(fill_value=0))

## 4. Error Type × Problem Type Heatmap

In [ ]:
ERROR_ORDER = ['correct','truncated','missing_marker','wrong_arithmetic',
               'hallucinated_calc','wrong_setup','off_by_one']
PTYPE_ORDER = list(PROBLEM_TYPES.keys()) + ['other']

fig, axes = plt.subplots(1, len(runs), figsize=(6*len(runs), 5), sharey=True)
if len(runs) == 1:
    axes = [axes]

for ax, (label, _) in zip(axes, runs.items()):
    df_last = df[(df['run'] == label) &
                 (df['step'] == df[df['run']==label]['step'].max()) &
                 (df['sample'] == 0)]   # use sample 0 for error analysis
    ct = pd.crosstab(df_last['error_type'], df_last['problem_type'])
    # Normalise by problem-type column so colour = fraction of that type with this error
    ct_norm = ct.div(ct.sum(axis=0), axis=1).fillna(0)
    # Reindex to fixed order
    ct_norm = ct_norm.reindex(
        index=[e for e in ERROR_ORDER if e in ct_norm.index],
        columns=[p for p in PTYPE_ORDER if p in ct_norm.columns]
    ).fillna(0)
    sns.heatmap(ct_norm, ax=ax, annot=True, fmt='.2f', cmap='YlOrRd',
                vmin=0, vmax=1, linewidths=0.4, cbar=(ax == axes[-1]))
    ax.set_title(f'{label}\n(last eval step)')
    ax.set_xlabel('Problem Type')
    ax.set_ylabel('Error Type' if ax == axes[0] else '')
    ax.tick_params(axis='x', rotation=35)

plt.suptitle('Error Type × Problem Type (fraction within each problem type)', y=1.02)
plt.tight_layout()
plt.savefig('error_x_problem_heatmap.png', bbox_inches='tight')
plt.show()

## 5. TF-IDF + K-Means Clustering of Incorrectly-Solved Problems

In [ ]:
N_CLUSTERS = 6

for label in runs:
    df_last = df[(df['run'] == label) &
                 (df['step'] == df[df['run']==label]['step'].max()) &
                 (df['sample'] == 0) &
                 (~df['is_correct'])]
    if len(df_last) < N_CLUSTERS:
        print(f'{label}: too few incorrect examples ({len(df_last)}) to cluster')
        continue

    texts = df_last['question'].tolist()
    vec   = TfidfVectorizer(max_features=500, stop_words='english', ngram_range=(1,2))
    X     = vec.fit_transform(texts).toarray()

    km = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init=10)
    labels_km = km.fit_predict(X)

    pca = PCA(n_components=2, random_state=42)
    X2  = pca.fit_transform(X)

    fig, ax = plt.subplots(figsize=(7, 5))
    scatter = ax.scatter(X2[:, 0], X2[:, 1], c=labels_km,
                         cmap='tab10', alpha=0.7, s=40, edgecolors='none')
    plt.colorbar(scatter, ax=ax, label='Cluster')
    ax.set_title(f'TF-IDF + K-Means ({N_CLUSTERS} clusters) — {label}\nIncorrect problems (PCA 2D)')
    ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})')
    ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%})')
    plt.tight_layout()
    plt.savefig(f'cluster_{label.lower().replace(" ","_")}.png', bbox_inches='tight')
    plt.show()

    # Top terms per cluster
    feature_names = vec.get_feature_names_out()
    print(f'\n── {label} cluster keywords ──')
    for c in range(N_CLUSTERS):
        center = km.cluster_centers_[c]
        top    = center.argsort()[-6:][::-1]
        kws    = ', '.join(feature_names[top])
        size   = (labels_km == c).sum()
        print(f'  Cluster {c} ({size:3d} problems): {kws}')

## 6. Difficulty Distribution — Question Length vs Accuracy

In [ ]:
fig, axes = plt.subplots(1, len(runs), figsize=(5.5*len(runs), 4), sharey=True)
if len(runs) == 1:
    axes = [axes]

for ax, (label, _) in zip(axes, runs.items()):
    df_last = df[(df['run'] == label) &
                 (df['step'] == df[df['run']==label]['step'].max()) &
                 (df['sample'] == 0)].copy()
    df_last['q_len']   = df_last['question'].str.len()
    df_last['ans_len'] = df_last['reference'].str.len()

    # Bin by question length
    df_last['q_bin'] = pd.cut(df_last['q_len'], bins=5)
    acc_by_len = df_last.groupby('q_bin')['is_correct'].mean()

    ax.bar(range(len(acc_by_len)),
           acc_by_len.values,
           color=COLORS.get(label, 'steelblue'),
           alpha=0.8, edgecolor='white')
    ax.set_xticks(range(len(acc_by_len)))
    ax.set_xticklabels([str(b) for b in acc_by_len.index], rotation=30, ha='right', fontsize=8)
    ax.set_title(f'{label}')
    ax.set_xlabel('Question Length (chars)')
    ax.set_ylabel('Pass@1 Accuracy')
    ax.set_ylim(0, 1)

plt.suptitle('Difficulty by Question Length — last eval step', y=1.02)
plt.tight_layout()
plt.savefig('difficulty_length.png', bbox_inches='tight')
plt.show()

# Scatter: q_len vs accuracy per problem
fig, ax = plt.subplots(figsize=(7, 4))
for label in runs:
    df_last = df[(df['run'] == label) &
                 (df['step'] == df[df['run']==label]['step'].max()) &
                 (df['sample'] == 0)].copy()
    df_last['q_len'] = df_last['question'].str.len()
    # jitter y slightly
    jitter = np.random.uniform(-0.02, 0.02, len(df_last))
    ax.scatter(df_last['q_len'], df_last['is_correct'].astype(float) + jitter,
               alpha=0.25, s=15, color=COLORS[label], label=label)
ax.set_xlabel('Question Length (chars)')
ax.set_ylabel('Correct (jittered)')
ax.set_title('Question Length vs Correctness')
ax.legend()
plt.tight_layout()
plt.savefig('length_vs_correctness_scatter.png', bbox_inches='tight')
plt.show()

## 7. Error Type Evolution Over Training Steps

In [ ]:
fig, axes = plt.subplots(len(runs), 1, figsize=(9, 4*len(runs)), sharex=False)
if len(runs) == 1:
    axes = [axes]

ERROR_COLORS = {
    'correct':           '#4CAF50',
    'truncated':         '#FF9800',
    'missing_marker':    '#F44336',
    'wrong_arithmetic':  '#2196F3',
    'hallucinated_calc': '#9C27B0',
    'wrong_setup':       '#795548',
    'off_by_one':        '#00BCD4',
}

for ax, (label, _) in zip(axes, runs.items()):
    df_run = df[(df['run'] == label) & (df['sample'] == 0)]
    steps  = sorted(df_run['step'].unique())
    bottom = np.zeros(len(steps))
    for etype in ERROR_ORDER:
        counts = []
        for st in steps:
            n = (df_run[df_run['step'] == st]['error_type'] == etype).sum()
            counts.append(n)
        counts = np.array(counts, dtype=float)
        totals = np.array([(df_run['step'] == st).sum() for st in steps], dtype=float)
        fracs  = np.divide(counts, totals, where=totals>0)
        ax.bar(steps, fracs, bottom=bottom,
               label=etype, color=ERROR_COLORS.get(etype, 'grey'), width=max(1, steps[-1]//30) if len(steps)>1 else 5)
        bottom += fracs
    ax.set_title(f'{label} — Error Type Evolution')
    ax.set_xlabel('RL Step')
    ax.set_ylabel('Fraction')
    ax.set_ylim(0, 1)
    ax.legend(loc='upper right', fontsize=8, ncol=2)

plt.tight_layout()
plt.savefig('error_evolution.png', bbox_inches='tight')
plt.show()

## 8. Per-Problem-Type Accuracy by Run

In [ ]:
records_last = []
for label in runs:
    df_last = df[(df['run'] == label) &
                 (df['step'] == df[df['run']==label]['step'].max()) &
                 (df['sample'] == 0)]
    for ptype, grp in df_last.groupby('problem_type'):
        records_last.append({'run': label, 'problem_type': ptype,
                             'accuracy': grp['is_correct'].mean(),
                             'n': len(grp)})

acc_df = pd.DataFrame(records_last)
pivot  = acc_df.pivot(index='problem_type', columns='run', values='accuracy').fillna(0)

ax = pivot.plot(kind='bar', figsize=(10, 4.5), color=[COLORS[r] for r in pivot.columns],
                edgecolor='white', width=0.7)
ax.set_title('Per-Problem-Type Accuracy by Reward System (last eval step)')
ax.set_ylabel('Pass@1 Accuracy')
ax.set_xlabel('')
ax.set_ylim(0, 1)
plt.xticks(rotation=30, ha='right')
plt.legend(title='Reward System')
plt.tight_layout()
plt.savefig('accuracy_by_problem_type.png', bbox_inches='tight')
plt.show()
print(pivot.round(3).to_string())

## 9. Summary Statistics

In [ ]:
print('=' * 65)
print(f'{"Run":<25} {"Last Step":>9} {"Pass@1":>8} {"Pass@8":>8}')
print('-' * 65)
for label in runs:
    df_run   = df[df['run'] == label]
    last     = df_run['step'].max()
    df_last  = df_run[df_run['step'] == last]
    steps_p1, vals_p1 = passk_curve(df_last, 1)
    steps_p8, vals_p8 = passk_curve(df_last, 8)
    p1 = vals_p1[0] if vals_p1 else float('nan')
    p8 = vals_p8[0] if vals_p8 else float('nan')
    print(f'{label:<25} {last:>9} {p1:>8.3f} {p8:>8.3f}')
print('=' * 65)

## 10. Commentary

### Reward System Design Rationale

**Baseline** uses exact-match only: reward = 1.0 iff the model outputs `#### <N>` matching the reference. For a model with near-zero initial accuracy this is extremely sparse — gradients carry no signal about *how wrong* an incorrect response is. The result is slow early learning and high variance.

**Numeric Distance** keeps exact-match (1.0) but adds dense shaping for near-misses:
```
reward = 0.10 * parseable + 0.35 * exp(−|pred−ref| / (|ref|+1))
```
A response with a plausible number gets non-zero credit proportional to its proximity to the reference. This provides informative gradients much earlier in training — the model learns to produce a parseable number before it learns the exact value. The cost is that the partial reward landscape may slow convergence to exact correctness if the model gets "comfortable" at ≈0.4 reward without ever reaching 1.0.

**Completion Brevity** also preserves exact-match (1.0) and adds:
```
reward = 0.20 * parseable + 0.10 * has_reasoning_words + 0.15 * brevity_score
```
where `brevity_score = 1.0` for outputs ≤ 220 chars, linearly decaying to 0.0 at 520 chars. The intent is to combat the model's tendency to ramble and fail to terminate with a `#### N` answer. By rewarding conciseness and structural markers, it pressures the policy toward short, well-formed responses. The tradeoff is that it rewards style over substance — the model can achieve ~0.45 reward with a short, structured but numerically wrong response.

### Observed Differences

*(Fill in after loading your actual data — the plots above will reveal this.)*

**Pass@k curves**: Numeric Distance is expected to show faster initial rise in pass@1 compared to Baseline due to denser gradient signal. Completion Brevity may show lower pass@8 if brevity pressure reduces sample diversity.

**Error type evolution**: Baseline likely stays high on `missing_marker` early (model never reaches `####`). Numeric Distance should show faster reduction in `wrong_arithmetic` errors as the reward explicitly penalises numeric distance. Completion Brevity should show faster reduction in `truncated` and `missing_marker` since brevity reward incentivises the model to terminate with a number.

**Problem-type accuracy**: All systems are expected to be weakest on geometry and fractions (multi-step spatial/symbolic reasoning) and strongest on counting and money problems (single arithmetic operation). If Numeric Distance has higher accuracy on wrong-arithmetic problems and lower on wrong-setup problems, this would confirm the hypothesis that proximity shaping helps once the model has the right setup but makes small arithmetic errors.

**Cluster analysis**: Incorrectly-solved problems tend to cluster around multi-step compound problems, unit-conversion chains, and problems with several embedded quantities. Clusters that are uniformly wrong across all three reward systems represent fundamental model capability limitations rather than reward-design issues.

**Difficulty by length**: Longer questions are generally harder. If Completion Brevity performs disproportionately worse on long questions, it may be because long questions require long chains of reasoning that the brevity penalty discourages.